# Chapter 15 — Multi-Agent Red/Blue 자동 공격·자동 방어

> **Chapter 15 · Part III**

공유 CVE corpus 를 근거로 레드팀 Agent 가 공격하고 블루팀 Agent 가
방어·자기학습(self-heal)하는 자동 공격·방어 루프를 구현합니다(Step 1~4). 이어서 §15.2~15.3 이
정의하는 **Red 3·Blue 3·Purple 1 의 7-Agent 구성**을 실제 모델(로컬 Ollama `llama3.2:3b` +
`claude-haiku-4-5` + `claude-opus-5`)로 구현합니다(Step 5~6).

## 실습 목표
1. 레드·블루가 **같은 CVE corpus** 를 공유하는 대칭 구조를 이해합니다.
2. 블루의 self-heal(놓친 공격을 corpus 로 되짚어 학습) 을 구현합니다.
3. RAG 사용/미사용을 비교해 CVE corpus 가 자기방어 향상에 기여함을 정량 확인합니다.
4. Red Pool(R1 정찰→R2 exploit→R3 횡적이동, Peer-to-Peer) 과 Blue Pool(B1 탐지·B2 분석[Actor-Critic]·
   B3 대응[Hierarchical]) 을 실제 LLM 호출로 구현합니다.
5. Purple Agent 가 두 로그를 종합해 평균 탐지 지연·Purple 합의율을 포함한 정량 보고서를
   만드는 과정을 구현합니다.

> **주의:** 모든 공격은 시뮬레이션입니다. 실제 exploit·스캔은 격리 VM·본인 자산에서만 수행합니다.
> Step 5~6 은 실제 네트워크 호출(로컬 Ollama, Anthropic API)을 사용합니다.

In [ ]:
# 첫 사용시에만
# !pip install -q anthropic ollama python-dotenv
# Step 5~6 은 실제 API/로컬 모델을 사용합니다 — 저장소 루트 .env 의 ANTHROPIC_API_KEY,
# 로컬에서 실행 중인 Ollama(`ollama serve`)와 `ollama pull llama3.2:3b` 가 필요합니다.

In [ ]:
import random, hashlib
rng = random.Random(42)   # 재현성

---
## Step 1. 공유 CVE corpus

레드의 공격 근거이자 블루의 방어 근거입니다(ch11 RAG 의 축소판). 실제로는 `labs/cve_db` 의
SQLite/벡터스토어를 씁니다.

In [ ]:
CVE_DB = {
    'CVE-2021-44228': {'name': 'Log4Shell',    'pattern': 'log4j jndi ldap rce'},
    'CVE-2017-5638':  {'name': 'Struts2',       'pattern': 'struts content-type ognl rce'},
    'CVE-2014-0160':  {'name': 'Heartbleed',    'pattern': 'openssl tls heartbeat disclosure'},
    'CVE-2022-22965': {'name': 'Spring4Shell',  'pattern': 'spring data binding classloader rce'},
    'CVE-2018-7600':  {'name': 'Drupalgeddon2', 'pattern': 'drupal form api render rce'},
}
def sig(cve_id):
    return ' '.join(sorted(CVE_DB[cve_id]['pattern'].split()))

# 격리 표적: 서비스마다 CVE 하나 (labs/target.py 와 대응)
TARGET = {'web-dvwa':'CVE-2021-44228','struts':'CVE-2017-5638','openssl':'CVE-2014-0160',
          'spring':'CVE-2022-22965','drupal':'CVE-2018-7600'}
print('corpus:', len(CVE_DB), '| 표적:', len(TARGET))

---
## Step 2. 레드팀 Agent — CVE 선택 → PoC(변형) → 공격

라운드가 오를수록 난독화 변형 비율을 높여 탐지를 회피합니다.

In [ ]:
def obfuscate(s):
    h = hashlib.sha1(s.encode()).hexdigest()[:6]
    return f'obf::{h}::' + s.replace(' ', '_')

# TODO: 아래에 코드를 작성하세요.
# 힌트: def red_attack(round_idx):

demo = red_attack(3)
print('공격 이벤트:', len(demo), '| 변형:', sum(e['mutated'] for e in demo))

<details><summary><b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가렸습니다.

```python
def red_attack(round_idx):
    ...
    mutate_prob = ...
    events = ...
    for service, cve_id in TARGET.items():
        base = ...
        mutated = ...
        payload = ...
        ...
    return ...
```
</details>


---
## Step 3. 블루팀 Agent — 탐지·차단 + self-heal

블루는 처음엔 일부 CVE 만 알고 난독화도 못 벗깁니다. 놓친 공격을 **공유 corpus 로 되짚어**
시그니처를 학습하고, 난독화를 한 번 보면 벗기는 법을 익힙니다.

In [ ]:
def normalize(payload, can_deobf):
    if can_deobf and payload.startswith('obf::'):
        return payload.split('::', 2)[2].replace('_', ' ')
    return payload

class BlueAgent:
    def __init__(self, known_cves, use_rag=True):
        self.use_rag = use_rag
        self.known = {sig(c) for c in known_cves}
        self.can_deobf = False

    # TODO: 아래에 코드를 작성하세요.
# 힌트: def defend(self, events):
print('BlueAgent 정의 완료')

<details><summary><b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가렸습니다.

```python
    def defend(self, events):
        detected, missed = ...
        for e in events:
            obs = ...
            if obs in self.known:
                detected += ...
            else:
                ...
        return ...

    def self_heal(self, missed):
        for e in missed:
            if e['mutated']:
                self.can_deobf = ...
            if self.use_rag: ...
                ...
```
</details>


---
## Step 4. 오케스트레이터(Purple) — 라운드 진행·점수

레드 공격 → 블루 방어 → 점수 → 블루 self-heal 을 반복하며 탐지율 곡선을 만듭니다.

In [ ]:
def run(rounds=7, use_rag=True):
    global rng; rng = random.Random(42)   # 라운드 재현성
    blue = BlueAgent(['CVE-2021-44228', 'CVE-2014-0160'], use_rag=use_rag)  # 5개 중 2개만 알고 시작
    hist = []
    for r in range(rounds):
        events = red_attack(r)
        detected, missed = blue.defend(events)
        rate = detected / len(events)
        blue.self_heal(missed)
        hist.append(rate)
    return hist

on  = run(use_rag=True)
off = run(use_rag=False)
for r,(a,b) in enumerate(zip(on, off), 1):
    bar = '#' * round(a*20)
    print(f'Day{r}: RAG {a*100:5.0f}% {bar:<20}  | 대조군 {b*100:.0f}%')
print(f'\n[결론] RAG 사용 {on[0]*100:.0f}%→{on[-1]*100:.0f}%, '
      f'대조군 {off[0]*100:.0f}%→{off[-1]*100:.0f}% '
      f'(공유 CVE corpus 기여 +{(on[-1]-off[-1])*100:.0f}%p)')

---
## Step 5. 7-Agent 구조 — Red Pool(R1→R2→R3) · Blue Pool(B1→B2→B3)

여기서부터는 §15.2~15.3 이 정의하는 **Red 3·Blue 3·Purple 1** 의 7-Agent 구성을 실제 모델로
구현합니다. Red Pool 은 로컬 Ollama `llama3.2:3b`, Blue Pool 은 `claude-haiku-4-5`,
Purple 은 `claude-opus-5`(실패 시 `claude-haiku-4-5` 폴백) 를 사용합니다. 위 Step 1~4 의
`CVE_DB`·`TARGET`·`sig()`·`obfuscate()` 를 그대로 재사용하므로, 두 영역이 같은 corpus 위에서
동작합니다.

호출 횟수를 bound 하기 위해 라운드 수는 작게(2 라운드 × 5 표적 = 10 이벤트) 유지합니다.
Step 4 의 7 라운드 탐지율 추세는 이미 확인했으므로, 여기서는 **7-Agent 역할 분리·통신 패턴·
Purple 정량 보고서** 를 보여주는 데 집중합니다.

> **주의:** 이 Step 은 실제 네트워크 호출을 합니다 — 로컬 Ollama(`ollama serve`, `llama3.2:3b` pull
> 완료)와 저장소 루트 `.env` 의 `ANTHROPIC_API_KEY` 가 필요합니다.

In [ ]:
import os, time, json, re
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))   # 저장소 루트 .env 의 ANTHROPIC_API_KEY 로드

import ollama
from anthropic import Anthropic

OLLAMA_MODEL = 'llama3.2:3b'                      # Red Pool (R1~R3)
BLUE_MODEL = 'claude-haiku-4-5'          # Blue Pool (B1~B3)
PURPLE_MODEL_PRIMARY = 'claude-opus-5'            # Purple (책 기준)
PURPLE_MODEL_FALLBACK = 'claude-haiku-4-5'

ollama_client = ollama.Client(host='http://localhost:11434')
claude_client = Anthropic()   # ANTHROPIC_API_KEY 환경변수 사용


def ollama_ask(prompt, num_predict=40):
    """Red Pool 공용 — 짧고 저렴한 Ollama 호출."""
    resp = ollama_client.generate(model=OLLAMA_MODEL, prompt=prompt,
                                   options={'num_predict': num_predict, 'temperature': 0.4})
    return resp['response'].strip()


def claude_text(msg):
    """모든 TextBlock 을 이어붙인다 — Opus 계열은 thinking 블록이 먼저 올 수 있어,
    첫 블록만 읽으면 빈 문자열을 얻는다."""
    return ''.join(b.text for b in msg.content if getattr(b, 'type', None) == 'text')


def claude_ask(prompt, model=BLUE_MODEL, max_tokens=1024, disable_thinking=False):
    """Blue/Purple 공용 — 짧은 Claude 호출.

    JSON 요약이 목적인 호출(Purple)은 disable_thinking=True 로 추론을 끈다. Opus 5 는
    thinking 이 기본 on 이고 추론 토큰도 max_tokens 에 포함돼, max_tokens 를 작게 주면
    예산을 추론이 다 써 text 블록이 비어 온다(그래서 예전에는 Purple 이 opus-5 에서
    파싱 실패로 haiku 폴백에 빠졌다). temperature 는 SDK 1.x 에서 인자로 넘기지 않는다."""
    kwargs = {'model': model, 'max_tokens': max_tokens,
              'messages': [{'role': 'user', 'content': prompt}]}
    if disable_thinking:
        kwargs['thinking'] = {'type': 'disabled'}
    msg = claude_client.messages.create(**kwargs)
    return claude_text(msg)


def parse_json_loose(text, default=None):
    m = re.search(r'\{.*\}', text, re.S)
    if not m:
        return default
    try:
        return json.loads(m.group(0))
    except json.JSONDecodeError:
        return default


print('Ollama:', OLLAMA_MODEL, '| Blue:', BLUE_MODEL,
      '| Purple:', PURPLE_MODEL_PRIMARY, '(폴백:', PURPLE_MODEL_FALLBACK + ')')

### Red Pool — R1(정찰) → R2(exploit 준비) → R3(횡적 이동), Peer-to-Peer 순차 통신 (§15.3.3)

각 단계가 독립 컨텍스트로 동작하고, 앞 단계의 출력만 다음 단계 입력이 됩니다. R2 는 신규
exploit 을 창작하지 않고 공유 CVE corpus 의 알려진 시그니처(`sig()`/`obfuscate()`)를
그대로 사용합니다(§15.2.2 참고). 표적·라운드마다 **난독화 강도**(none/light/strong)를 달리해,
강한 난독화(`obfuscate_strong()`)는 원본 키워드를 남기지 않아 Blue 의 시그니처 탐지를 실제로
어렵게 만듭니다. R3 는 자신의 침투가 성공했을지 스스로 평가해 자기보고(`self_report_success`)를
남기는데, 이는 Blue 의 실제 판정과 **별개**의 값입니다 — Red 는 자신이 우회했다고 낙관하지만
Blue 가 그 페이로드를 실제로 잡아낼 수 있어, 이 어긋남이 뒤의 Purple 합의율 지표가 됩니다.

In [ ]:
# TODO: 아래에 코드를 작성하세요.
# 힌트: 라운드·표적별 난독화 강도 — 강도가 오를수록 Blue 시그니처 탐지가 어려워진다(§15.3.3).

_demo = red_pipeline(1, 'web-dvwa', 'CVE-2021-44228')
print('R1:', _demo['r1'][:70])
print('R2:', _demo['r2'][:70])
print('R3 자기보고:', 'SUCCESS' if _demo['self_report_success'] else 'DETECTED_RISK',
      '| 난독화:', _demo['obf_level'])

<details><summary><b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가렸습니다.

```python
# 라운드·표적별 난독화 강도 — 강도가 오를수록 Blue 시그니처 탐지가 어려워진다(§15.3.3).
# 표적을 다양화하고 강도를 섞어, Red 자기보고와 Blue 실제 판정이 자연스럽게 엇갈리게 한다.
OBF_LEVELS = ...
    ...


def obfuscate_strong(s):
    ...
    return ...


def red_r1_recon(service, cve_name):
    ...
    prompt = ...
              ...
    return ...


def red_r2_exploit(recon_note, sig_hint):
    ...
    prompt = ...
              ...
    return ...


def red_r3_lateral(exploit_note, level):
    ...
    hint = ...
        ...
    prompt = ...
              ...
    verdict = ...
    return ...


def red_pipeline(round_idx, service, cve_id):
    ...
    t_start = ...
    cve = ...
    r1_note = ...

    base = ...
    level = ...
    if level == 'strong':
        payload = ...
    elif level == 'light':
        payload = ...
    else:
        payload = ...
    r2_note = ...

    self_report_success = ...
    t_r3_done = ...

    return ...
        ...
```
</details>


### Blue Pool — B1(탐지) · B2(분석, Actor-Critic) · B3(대응, Hierarchical)

B2 는 **Actor-Critic** 패턴(§15.3.2)입니다 — actor 가 분석을 내고, critic 이 같은 모델·다른
prompt 로 과장·환각 여부를 검증합니다. 문제가 있으면 확신도를 보수적으로 낮춥니다(비용을
위해 별도의 재작성 호출은 생략).

B3 는 **Hierarchical** 패턴(§15.3.4)입니다 — `blue_b3_respond()` 함수 본문이 `blue_b1_detect()`
와 `blue_b2_analyze()` 를 직접 호출하고, B2 의 확신도(confidence)를 기준으로 책의 의사결정
트리(> 0.9 block, 0.5~0.9 escalate, 그 외 pass)를 그대로 적용합니다.

In [ ]:
# TODO: 아래에 코드를 작성하세요.
# 힌트: 자기 검증에서 문제 발견 시 확신도를 보수적으로 낮춘다 (재작성 호출은 비용상 생략)
print('B1/B2/B3 정의 완료 (Actor-Critic + Hierarchical)')

<details><summary><b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가렸습니다.

```python
def blue_b1_detect(payload, corpus_lines):
    ...
    prompt = ...
              ...
    text = ...
    return ...


def blue_b2_analyze(detection, payload):
    ...
    actor_prompt = ...
                     ...
    actor_text = ...
    analysis = ...

    critic_prompt = ...
                      ...
    critic_text = ...
    critique = ...

    if critique.get('has_issues'):
        # 자기 검증에서 문제 발견 시 확신도를 보수적으로 낮춘다 (재작성 호출은 비용상 생략)
        analysis['confidence'] = ...
    return ...


def blue_b3_respond(payload, corpus_lines):
    ...
    detection = ...
    t_detected = ...
    analysis, critique = ...

    confidence = ...
    if confidence > 0.9:
        action = ...
    elif confidence > 0.5:
        action = ...
    else:
        action = ...

    return ...
            ...
```
</details>


---
## Step 6. 7-Agent 루프 실행 + Purple Agent 정량 보고서

2 라운드 × 5 표적 = 10 이벤트를 Red→Blue 파이프라인으로 돌리고 로그를 모읍니다. 그다음
Purple Agent 가 두 로그를 종합해 §15.5 의 지표 중 **평균 Red→Blue 탐지 지연**과
**Purple 합의율(Red 자기보고 vs Blue 실제 판정 일치율)** 을 계산·보고합니다.

In [ ]:
# 공유 corpus 를 Blue 의 탐지 근거 컨텍스트로 사용 (Step 3 의 self.known 과 같은 개념)
CORPUS_LINES = '\n'.join(f"- {c['name']} ({cid}): {c['pattern']}" for cid, c in CVE_DB.items())

ROUNDS_DEMO = 2   # API 호출 총량을 bound (Step4 의 7라운드 탐지율 추세는 이미 확인함)

red_log, blue_log = [], []
for r in range(ROUNDS_DEMO):
    print(f'--- Round {r + 1} ---')
    for service, cve_id in TARGET.items():
        red_ev = red_pipeline(r, service, cve_id)
        blue_ev = blue_b3_respond(red_ev['payload'], CORPUS_LINES)
        red_log.append(red_ev)
        blue_log.append(blue_ev)

        red_tag = 'SUCCESS' if red_ev['self_report_success'] else 'DETECTED_RISK'
        detected = bool(blue_ev['detection'].get('detected'))
        print(f"  R[{service:10s}] {red_tag:14s} obf={red_ev['obf_level']:6s} -> "
              f"B detected={detected!s:5s} conf={blue_ev['analysis'].get('confidence')} "
              f"action={blue_ev['action']}")

print(f'\n총 {len(red_log)} 이벤트 수집 '
      f'(Ollama 호출 {len(red_log) * 3}회, Claude 호출 {len(blue_log) * 3}회)')

In [ ]:
# TODO: 아래에 코드를 작성하세요.
# 힌트: 합의: Red 의 자기보고(성공/탐지위험)와 Blue 의 실제 판정(미탐/탐지)이 일치하는가

report = purple_report(red_log, blue_log)
print(json.dumps(report, ensure_ascii=False, indent=2))

<details><summary><b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가렸습니다.

```python
def purple_report(red_log, blue_log):
    ...

    ...
    요청을 거부(stop_reason = ...
    ...
    lags, consensus_hits, disagreements = ...
    for red_ev, blue_ev in zip(red_log, blue_log):
        detected = ...
        if detected:
            ...
        # 합의: Red 의 자기보고(성공/탐지위험)와 Blue 의 실제 판정(미탐/탐지)이 일치하는가
        agree = ...
        ...
        if not agree:
            tag = ...
            verdict = ...
            ...

    n = ...
    red_success_n = ...
    blue_detect_n = ...
    red_success_rate = ...
    blue_detection_rate = ...
    avg_lag = ...
    consensus_rate = ...
    lag_text = ...
    dis_services = ...

    # 중립적 '탐지 품질 감사' 어휘로 구성(공격 어휘 배제). 총평 2문장만 요청한다.
    prompt = ...
        ...

    model_used = ...
    summary = ...
    if not summary.strip(): ...
        ...
        model_used = ...
        summary = ...

    # 수치는 코드 집계값이 정본이다. 총평 문장만 LLM 산출.
    return ...
        ...
```
</details>


---
## 정리

- 레드와 블루가 **같은 CVE corpus** 를 공유하므로, 블루는 놓친 공격을 corpus 로 되짚어 스스로
  방어를 고도화할 수 있습니다(self-heal). RAG(공유 corpus) 없이는 미지 변형 공격을 학습하지
  못해 탐지율이 평탄합니다(Step 1~4).
- Red Pool(R1 정찰→R2 exploit 준비→R3 횡적이동, Peer-to-Peer 순차 통신)과 Blue Pool(B1 탐지·
  B2 분석[Actor-Critic 자기검증]·B3 대응[Hierarchical, B1·B2 를 직접 호출])을 실제 Ollama·
  Claude 호출로 구현했습니다(Step 5).
- Purple Agent 가 Red·Blue 로그를 종합해 평균 Red→Blue 탐지 지연과 Purple 합의율(Red 자기보고
  vs Blue 실제 판정의 일치율)을 계산·보고합니다(Step 6). 이 두 지표가 §15.5 정량 보고서의
  핵심 항목입니다.
- 실제 격리 표적·MCP 도구로 확장한 실행체는 저장소 `labs/`(`python labs/run_lab.py`) 에 있습니다.

> **다음 챕터:** LLM-Native SIEM — Multi-Agent 운영의 텔레메트리·상관·검색을 LLM 친화적으로 재설계.